# SafePub

SafePub provides ($\varepsilon$, $\delta$)-differential privacy through random sampling, full-domain generalization, and record suppression [1].

The privacy parameters ($\varepsilon$, $\delta$) are converted into a sampling probability $\beta$ and a class-size threshold $k$. The algorithm draws a random $\beta$-sample of the records and releases only the sampled records whose equivalence classes contain at least $k$ records; everything else is suppressed.

In the **data-dependent** mode (default), a fraction of the privacy budget (10% by default) is spent on selecting the generalization levels with the exponential mechanism. In the **data-independent** mode, the generalization levels are fixed up front and the full budget is spent on anonymization.

Unlike the other algorithms in this package, SafePub is **randomized**: the released subset — and, in the data-dependent mode, the chosen generalization — differ between runs by design. Note also that `k` is not a free parameter: it is derived from ($\varepsilon$, $\delta$).

[1] Bild, Raffael, Klaus A. Kuhn, and Fabian Prasser. “SafePub: A Truthful Data Anonymization Algorithm With Strong Privacy Guarantees.” Proceedings on Privacy Enhancing Technologies 2018.1 (2018): 67–87. https://doi.org/10.1515/popets-2018-0004

In [ ]:
from k_anonymization import datasets
from k_anonymization.algorithms.full_generalization import SafePub

## Data-dependent SafePub

Only $\varepsilon$ and $\delta$ are required ($\delta$ should be smaller than `1 / number_of_records`). The sampling probability $\beta$ and the class-size threshold $k$ are derived from them.

In [ ]:
safepub = SafePub(
    dataset=datasets.ADULT,
    epsilon=2.0,
    delta=1e-5,
    search_expansion_limit=20,
)
print(f"k = {safepub.k}, beta = {safepub.beta:.6f}")

In [ ]:
safepub.anonymize()

print("generalization levels:", safepub.levels)
print(f"score ({safepub.score_function}, higher is better):", safepub.score)
print("sampled records:", safepub.sampled_count)
print("suppressed sampled records (class < k):", safepub.suppressed_sample_count)
print("released records:", len(safepub.anon_data))

In [ ]:
safepub.anon_data

The suppressed equivalence classes are available in `suppressed_qids`, like for the other algorithms:

In [ ]:
safepub.suppressed_qids[:5]

## Choosing the quality model

The `utility_metric` parameter selects the quality model whose SafePub score function drives the data-dependent search. All six models supported by ARX are available: `safepub_precision` (default), `safepub_loss`, `safepub_discernibility`, `safepub_entropy`, `safepub_aecs`, and `safepub_classification` (the latter requires `response_variables`).

In [ ]:
safepub_dm = SafePub(
    dataset=datasets.ADULT,
    epsilon=2.0,
    delta=1e-5,
    search_expansion_limit=20,
    utility_metric="safepub_discernibility",
)
safepub_dm.anonymize()

print("generalization levels:", safepub_dm.levels)
print(f"score ({safepub_dm.score_function}):", safepub_dm.score)
print("released records:", len(safepub_dm.anon_data))

## Data-independent SafePub (fixed scheme)

With `data_dependent=False` the generalization levels are fixed up front — per attribute via `generalization_levels`, and/or for the remaining attributes via `generalization_degree` (`none`, `low`, `low_medium`, `medium`, `medium_high`, `high`, `complete`) — and the full $\varepsilon$ is spent on anonymization.

In [ ]:
safepub_fixed = SafePub(
    dataset=datasets.ADULT,
    epsilon=2.0,
    delta=1e-5,
    data_dependent=False,
    generalization_degree="medium",
    generalization_levels={"age": 2},
)
safepub_fixed.anonymize()

print("generalization levels:", safepub_fixed.levels)
print("released records:", len(safepub_fixed.anon_data))